## Combined evaluation (E1–E4) — Kaggle-ready, self-contained

Implements the evaluation protocol from `technical-specification.md` in a **single notebook**.

### Kaggle datasets required
All datasets must be attached under:
`/kaggle/input/datasets/karandeepshoker/<dataset-slug>/...`

Required dataset slugs:
- `plain_v3/ckpt.pt`
- `simple_v2/ckpt.pt`
- `vae_v3/ckpt.pt`
- `preprocessed-memmaptest/` (contains the test memmaps)
- `misc/` (contains `tokenizer_meta.json` and `quantiles.json`)

Outputs are written under `/kaggle/working/artifacts/eval/`.


In [ ]:
from __future__ import annotations

import os
import json
import math
import time
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

print('cwd:', os.getcwd())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
OUT_DIR = WORKING_ROOT / 'artifacts' / 'eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if INPUT_ROOT.exists():
    print('--- /kaggle/input ---')
    for p in sorted(INPUT_ROOT.iterdir()):
        if p.is_dir():
            print(' -', p.name)


In [ ]:
# -------------------------
# Config (robust defaults)
# -------------------------

KAGGLE_USER = 'karandeepshoker'

PLAIN_RUN = 'plain_v3'
SIMPLE_RUN = 'simple_v2'
VAE_RUN = 'vae_v3'

TEST_DATASET = 'preprocessed-memmaptest'
# test split location inside that dataset (follow training notebooks convention)
TEST_SPLIT_REL_CANDIDATES = [Path('test'), Path('.')]

MISC_DATASET = 'misc'
TOKENIZER_META_NAME = 'tokenizer_meta.json'
QUANTILES_NAME = 'quantiles.json'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# E1
E1_BATCH_SIZE = 16
E1_MAX_BATCHES = 200

# E2 (robust but time-bounded)
# This notebook will auto-downscale E2 per model to fit E2_TIME_BUDGET_SECS.
E2_N_PER_BIN = 100
E2_MIN_N_PER_BIN = 10
E2_TIME_BUDGET_SECS = 60 * 60  # target ~1 hour for E2 total per model (plain is slowest)
E2_BASELINE_BIN = 3
E2_TEMPERATURE = 1.0
E2_TOP_P = 0.9
E2_SEED = 0

# E3 (robust)
E3_N_LATENT = 20_000
E3_BATCH_SIZE = 64
E3_KNN_K = 15

# E4
E4_N_ANCHORS = 5
E4_N_NEAR = 40
E4_SIGMA = 0.15
E4_N_INTERP = 9

print('OUT_DIR:', OUT_DIR)
print('DEVICE:', DEVICE)


In [ ]:
# -------------------------
# Locate dataset paths (exactly like 04/05)
# -------------------------

def require_dir(p: Path) -> Path:
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(f'missing directory: {p}')
    return p


def require_file(p: Path) -> Path:
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f'missing file: {p}')
    return p


BASE = require_dir(Path('/kaggle/input/datasets') / KAGGLE_USER)

PLAIN_CKPT = require_file(require_dir(BASE / PLAIN_RUN) / 'ckpt.pt')
SIMPLE_CKPT = require_file(require_dir(BASE / SIMPLE_RUN) / 'ckpt.pt')
VAE_CKPT = require_file(require_dir(BASE / VAE_RUN) / 'ckpt.pt')

MISC_DIR = require_dir(BASE / MISC_DATASET)
TOKENIZER_META_PATH = require_file(MISC_DIR / TOKENIZER_META_NAME)
QUANTILES_PATH = require_file(MISC_DIR / QUANTILES_NAME)

TEST_ROOT = require_dir(BASE / TEST_DATASET)
TEST_DIR = None
for rel in TEST_SPLIT_REL_CANDIDATES:
    cand = TEST_ROOT / rel
    if cand.exists() and (cand / 'meta.json').exists():
        TEST_DIR = cand
        break
if TEST_DIR is None:
    raise FileNotFoundError(f'Could not find test split in {TEST_ROOT} (tried {TEST_SPLIT_REL_CANDIDATES})')

for f in ['meta.json', 'X.u16.memmap', 'bar_indices.u8.memmap', 'attributes.u8.memmap']:
    require_file(Path(TEST_DIR) / f)

print('PLAIN_CKPT:', PLAIN_CKPT)
print('SIMPLE_CKPT:', SIMPLE_CKPT)
print('VAE_CKPT:', VAE_CKPT)
print('TEST_DIR:', TEST_DIR)
print('TOKENIZER_META_PATH:', TOKENIZER_META_PATH)
print('QUANTILES_PATH:', QUANTILES_PATH)


In [ ]:
# -------------------------
# Self-contained code (write a tiny module file) — same pattern as 04/05
# -------------------------

import sys
import importlib

MODULE_PATH = Path('musicgen_kaggle_eval_all_min.py')

# --- Model code (verbatim from repo) ---
PLAIN_MODEL_CODE = r'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class PlainTransformerConfig:
    vocab_size: int = 195
    block_size: int = 1024
    n_layers: int = 6
    n_embed: int = 512
    n_head: int = 8
    d_ff: int = 2048
    bars_per_sample: int = 8
    dropout: float = 0.1

    # Attributes: 4 dims (polyphony, intensity, velocity, density), each binned 0..7
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128


class AttributeConditioner(nn.Module):
    """
    attributes: LongTensor [B, bars, 4] with values in 0..7
    returns:    FloatTensor [B, bars, n_embed]
    """

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        assert cfg.attribute_embed_dim * cfg.num_attributes == 128

        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )
        self.proj = nn.Linear(cfg.attribute_embed_dim * cfg.num_attributes, cfg.n_embed)

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        # attributes: [B, bars, 4]
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        A = torch.cat([a0, a1, a2, a3], dim=-1)  # [B, bars, 128]
        return self.proj(A)  # [B, bars, n_embed]


class PlainCausalSelfAttention(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.n_embed % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embed // cfg.n_head

        self.qkv = nn.Linear(cfg.n_embed, 3 * cfg.n_embed, bias=True)
        self.proj = nn.Linear(cfg.n_embed, cfg.n_embed, bias=True)
        self.dropout = float(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, C]
        B, T, C = x.shape
        qkv = self.qkv(x)  # [B, T, 3C]
        q, k, v = qkv.split(C, dim=-1)

        # [B, nh, T, hs]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # PyTorch uses optimized SDPA under the hood when available.
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )  # [B, nh, T, hs]
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # [B, T, C]
        return self.proj(y)


class PlainFiLMBlock(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embed)
        self.attn = PlainCausalSelfAttention(cfg)
        self.resid_drop = nn.Dropout(cfg.dropout)

        # Conditioned FiLM: gamma/beta per token timestep, derived from cond_tok [B,T,C]
        self.film = nn.Linear(cfg.n_embed, 2 * cfg.n_embed, bias=True)

        self.ln2 = nn.LayerNorm(cfg.n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embed, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.n_embed),
        )
        self.mlp_drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        # Pre-norm
        h = self.ln1(x)
        gamma_beta = self.film(cond_tok)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))

        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class PlainTransformerDecoder(nn.Module):
    """Decoder-only baseline conditioned ONLY on per-bar attributes via FiLM."""

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embed)

        self.attr = AttributeConditioner(cfg)
        self.blocks = nn.ModuleList([PlainFiLMBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.n_embed)
        self.head = nn.Linear(cfg.n_embed, cfg.vocab_size, bias=True)

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
    ) -> dict:
        B, T = X.shape
        if T > self.cfg.block_size:
            raise ValueError(f"Sequence length T={T} exceeds block_size={self.cfg.block_size}")

        # Token + position embeddings
        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]

        # Build per-bar conditions and broadcast to tokens using bar_indices
        cond_bars = self.attr(attributes)  # [B, bars, C]
        idx = bar_indices.to(dtype=torch.long).clamp_(0, self.cfg.bars_per_sample - 1)
        cond_tok = cond_bars.gather(1, idx.unsqueeze(-1).expand(-1, -1, self.cfg.n_embed))  # [B, T, C]

        for blk in self.blocks:
            x = blk(x, cond_tok)

        x = self.ln_f(x)
        logits = self.head(x)  # [B, T, vocab]

        out = {"logits": logits}
        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing loss with targets.")

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            n_tokens = int((targets != int(pad_id)).sum().item())
            out.update({"loss": loss, "n_tokens": n_tokens})

        return out
'''

SIMPLE_MODEL_CODE = r'''
from dataclasses import dataclass
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class SimpleVAEConfig:
    vocab_size: int = 195
    block_size: int = 1024

    # Encoder (bidirectional)
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1

    # Attributes
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128

    # Simple global latent projection (no Conductor)
    z_global_dim: int = 384

    # Decoder (causal)
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1


class AttributeEmbedder(nn.Module):
    """
    attributes: LongTensor [B, bars, 4] with values 0..7
    returns:    FloatTensor [B, bars, 128]
    """

    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        return torch.cat([a0, a1, a2, a3], dim=-1)  # [B, bars, 128]


def broadcast_by_bar_indices(bars_x_c: torch.Tensor, bar_indices: torch.Tensor) -> torch.Tensor:
    """bars_x_c: [B,bars,C], bar_indices: [B,T] -> [B,T,C]"""
    B, bars, C = bars_x_c.shape
    idx = bar_indices.to(dtype=torch.long).clamp_(0, bars - 1)
    return bars_x_c.gather(1, idx.unsqueeze(-1).expand(B, idx.size(1), C))


class BiTransformerEncoder(nn.Module):
    """Bidirectional encoder with learned [CLS] token, outputs mu/logvar for z_p."""

    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)  # +1 for CLS position
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.enc_n_embed,
            nhead=cfg.enc_n_head,
            dim_feedforward=cfg.enc_d_ff,
            dropout=float(cfg.enc_dropout),
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.ln_f = nn.LayerNorm(cfg.enc_n_embed)

        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)

        nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Encoder expects block_size={self.cfg.block_size}, got T={T}")

        tok = self.tok_emb(X)  # [B,T,C]
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)  # [B,1,C]
        h = torch.cat([cls, tok], dim=1)  # [B,T+1,C]

        pos = torch.arange(T + 1, device=X.device)
        h = h + self.pos_emb(pos)[None, :, :]

        h = self.encoder(h)
        h = self.ln_f(h)
        h_cls = h[:, 0, :]  # [B,C]
        return self.mu(h_cls), self.logvar(h_cls)


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed: int, n_head: int):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=True)
        self.proj = nn.Linear(n_embed, n_embed, bias=True)
        self.dropout = 0.0

    def set_dropout(self, p: float) -> None:
        self.dropout = float(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class FiLMBlock(nn.Module):
    def __init__(self, n_embed: int, n_head: int, d_ff: int, *, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed=n_embed, n_head=n_head)
        self.attn.set_dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.film = nn.Linear(n_embed, 2 * n_embed, bias=True)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, n_embed),
        )
        self.mlp_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class FiLMDecoder(nn.Module):
    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)

        self.blocks = nn.ModuleList(
            [
                FiLMBlock(cfg.dec_n_embed, cfg.dec_n_head, cfg.dec_d_ff, dropout=float(cfg.dec_dropout))
                for _ in range(cfg.dec_n_layers)
            ]
        )
        self.ln_f = nn.LayerNorm(cfg.dec_n_embed)
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size, bias=True)

    def forward(self, X: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Decoder expects block_size={self.cfg.block_size}, got T={T}")

        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        return self.head(x)  # [B,T,vocab]


class SimpleMusicVAE(nn.Module):
    """Simple (non-hierarchical) conditional VAE."""

    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        self.cfg = cfg
        self.attr = AttributeEmbedder(cfg)  # [B,8,128]
        self.encoder = BiTransformerEncoder(cfg)
        self.decoder = FiLMDecoder(cfg)

        self.z_proj = nn.Linear(cfg.z_dim, cfg.z_global_dim)

        cond_dim = int(cfg.z_global_dim) + int(cfg.attribute_embed_dim * cfg.num_attributes)
        if cond_dim != cfg.dec_n_embed:
            raise ValueError(f"Expected z_global_dim+attr_dim == dec_n_embed; got {cond_dim} vs {cfg.dec_n_embed}")

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    @staticmethod
    def kl_divergence_free_bits(
        mu: torch.Tensor,
        logvar: torch.Tensor,
        *,
        free_bits_lambda_bits: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        kl_per_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
        kl_per_dim_mean = kl_per_dim.mean(dim=0)
        lambda_nats = float(free_bits_lambda_bits) * math.log(2.0)
        kl_per_dim_hinged_mean = torch.clamp(kl_per_dim_mean, min=lambda_nats)
        kl_loss = kl_per_dim_hinged_mean.sum()
        active_units = (kl_per_dim_mean > lambda_nats).sum()
        return kl_loss, kl_per_dim_mean, active_units

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        z_p: Optional[torch.Tensor] = None,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
        beta: float = 1.0,
    ) -> dict:
        mu: Optional[torch.Tensor]
        logvar: Optional[torch.Tensor]
        if z_p is None:
            mu, logvar = self.encoder(X)
            z_p = self.reparameterize(mu, logvar)
        else:
            mu, logvar = None, None
            if z_p.ndim != 2 or z_p.shape[0] != X.shape[0]:
                raise ValueError(f"z_p must have shape [B,z_dim], got {tuple(z_p.shape)}")
            if z_p.shape[1] != self.cfg.z_dim:
                raise ValueError(f"z_p dim mismatch: expected {self.cfg.z_dim}, got {z_p.shape[1]}")

        z_g = self.z_proj(z_p)
        z_g_bars = z_g[:, None, :].expand(-1, self.cfg.bars_per_sample, -1)

        A_k = self.attr(attributes)
        C_k = torch.cat([z_g_bars, A_k], dim=-1)
        cond_tok = broadcast_by_bar_indices(C_k, bar_indices)

        logits = self.decoder(X, cond_tok)

        out = {"logits": logits, "mu": mu, "logvar": logvar, "z_p": z_p, "z_g": z_g}

        if mu is not None and logvar is not None:
            loss_kl, kl_per_dim_mean, active_units = self.kl_divergence_free_bits(mu, logvar, free_bits_lambda_bits=1.0)
            out["loss_kl"] = loss_kl
            out["kl_per_dim_mean"] = kl_per_dim_mean
            out["active_units"] = active_units

        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing reconstruction loss.")
            if mu is None or logvar is None:
                raise ValueError("targets provided but z_p was passed explicitly; KL stats are undefined in this mode.")
            loss_recon = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            out["loss_recon"] = loss_recon
            out["loss_total"] = loss_recon + float(beta) * loss_kl

        return out
'''

VAE_MODEL_CODE = r'''
from dataclasses import dataclass
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class VAEConfig:
    vocab_size: int = 195
    block_size: int = 1024

    # Encoder (bidirectional)
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1

    # Attributes
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128

    # Conductor
    cond_hidden: int = 384
    cond_layers: int = 2

    # Decoder (causal)
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1


class AttributeEmbedder(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        return torch.cat([a0, a1, a2, a3], dim=-1)


def broadcast_by_bar_indices(bars_x_c: torch.Tensor, bar_indices: torch.Tensor) -> torch.Tensor:
    B, bars, C = bars_x_c.shape
    idx = bar_indices.to(dtype=torch.long).clamp_(0, bars - 1)
    return bars_x_c.gather(1, idx.unsqueeze(-1).expand(B, idx.size(1), C))


class BiTransformerEncoder(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.enc_n_embed,
            nhead=cfg.enc_n_head,
            dim_feedforward=cfg.enc_d_ff,
            dropout=float(cfg.enc_dropout),
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.ln_f = nn.LayerNorm(cfg.enc_n_embed)

        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)

        nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Encoder expects block_size={self.cfg.block_size}, got T={T}")

        tok = self.tok_emb(X)
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)
        h = torch.cat([cls, tok], dim=1)

        pos = torch.arange(T + 1, device=X.device)
        h = h + self.pos_emb(pos)[None, :, :]

        h = self.encoder(h)
        h = self.ln_f(h)
        h_cls = h[:, 0, :]
        return self.mu(h_cls), self.logvar(h_cls)


class Conductor(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.h0_proj = nn.Linear(cfg.z_dim, cfg.cond_layers * cfg.cond_hidden)
        self.bar_emb = nn.Embedding(cfg.bars_per_sample, cfg.cond_hidden)
        self.gru = nn.GRU(
            input_size=cfg.cond_hidden,
            hidden_size=cfg.cond_hidden,
            num_layers=cfg.cond_layers,
            batch_first=True,
        )

    def forward(self, z_p: torch.Tensor) -> torch.Tensor:
        B = z_p.size(0)
        h0 = self.h0_proj(z_p).view(self.cfg.cond_layers, B, self.cfg.cond_hidden).contiguous()

        bar_ids = torch.arange(self.cfg.bars_per_sample, device=z_p.device)
        x = self.bar_emb(bar_ids)[None, :, :].expand(B, -1, -1)

        z_seq, _hn = self.gru(x, h0)
        return z_seq


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed: int, n_head: int):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=True)
        self.proj = nn.Linear(n_embed, n_embed, bias=True)
        self.dropout = 0.0

    def set_dropout(self, p: float) -> None:
        self.dropout = float(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class FiLMBlock(nn.Module):
    def __init__(self, n_embed: int, n_head: int, d_ff: int, *, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed=n_embed, n_head=n_head)
        self.attn.set_dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.film = nn.Linear(n_embed, 2 * n_embed, bias=True)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, n_embed),
        )
        self.mlp_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class FiLMDecoder(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)

        self.blocks = nn.ModuleList(
            [
                FiLMBlock(cfg.dec_n_embed, cfg.dec_n_head, cfg.dec_d_ff, dropout=float(cfg.dec_dropout))
                for _ in range(cfg.dec_n_layers)
            ]
        )
        self.ln_f = nn.LayerNorm(cfg.dec_n_embed)
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size, bias=True)

    def forward(self, X: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Decoder expects block_size={self.cfg.block_size}, got T={T}")

        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        return self.head(x)


class MusicVAE(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.attr = AttributeEmbedder(cfg)
        self.encoder = BiTransformerEncoder(cfg)
        self.conductor = Conductor(cfg)
        self.decoder = FiLMDecoder(cfg)

        if cfg.cond_hidden + (cfg.attribute_embed_dim * cfg.num_attributes) != cfg.dec_n_embed:
            raise ValueError(
                f"Expected cond_hidden(={cfg.cond_hidden}) + attr_dim(={cfg.attribute_embed_dim*cfg.num_attributes}) == dec_n_embed(={cfg.dec_n_embed})"
            )

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    @staticmethod
    def kl_divergence_free_bits(
        mu: torch.Tensor,
        logvar: torch.Tensor,
        *,
        free_bits_lambda_bits: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        kl_per_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
        kl_per_dim_mean = kl_per_dim.mean(dim=0)
        lambda_nats = float(free_bits_lambda_bits) * math.log(2.0)
        kl_per_dim_hinged_mean = torch.clamp(kl_per_dim_mean, min=lambda_nats)
        kl_loss = kl_per_dim_hinged_mean.sum()
        active_units = (kl_per_dim_mean > lambda_nats).sum()
        return kl_loss, kl_per_dim_mean, active_units

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        z_p: Optional[torch.Tensor] = None,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
        beta: float = 1.0,
    ) -> dict:
        mu: Optional[torch.Tensor]
        logvar: Optional[torch.Tensor]
        if z_p is None:
            mu, logvar = self.encoder(X)
            z_p = self.reparameterize(mu, logvar)
        else:
            mu, logvar = None, None
            if z_p.ndim != 2 or z_p.shape[0] != X.shape[0]:
                raise ValueError(f"z_p must have shape [B,z_dim], got {tuple(z_p.shape)}")
            if z_p.shape[1] != self.cfg.z_dim:
                raise ValueError(f"z_p dim mismatch: expected {self.cfg.z_dim}, got {z_p.shape[1]}")

        z_k = self.conductor(z_p)
        A_k = self.attr(attributes)
        C_k = torch.cat([z_k, A_k], dim=-1)
        cond_tok = broadcast_by_bar_indices(C_k, bar_indices)

        logits = self.decoder(X, cond_tok)
        out = {"logits": logits, "mu": mu, "logvar": logvar, "z_p": z_p, "z_k": z_k}

        if mu is not None and logvar is not None:
            loss_kl, kl_per_dim_mean, active_units = self.kl_divergence_free_bits(mu, logvar, free_bits_lambda_bits=1.0)
            out["loss_kl"] = loss_kl
            out["kl_per_dim_mean"] = kl_per_dim_mean
            out["active_units"] = active_units

        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing reconstruction loss.")
            if mu is None or logvar is None:
                raise ValueError("targets provided but z_p was passed explicitly; KL stats are undefined in this mode.")
            loss_recon = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            out["loss_recon"] = loss_recon
            out["loss_total"] = loss_recon + float(beta) * loss_kl

        return out
'''

# --- Memmap dataset (verbatim from 04/05 style) ---
DATA_CODE = r'''
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
from torch.utils.data import Dataset


PathLike = str | os.PathLike[str]


@dataclass(frozen=True)
class MemmapSplitMeta:
    split: str
    num_examples: int
    tokens_per_example: int
    bars_per_sample: int
    num_attributes: int
    pad_id: int
    bar_id: int


class MemmapMusicDataset(Dataset):
    def __init__(self, split_dir: PathLike):
        self.split_dir = Path(split_dir)
        payload = json.loads((self.split_dir / 'meta.json').read_text(encoding='utf-8'))

        self.meta = MemmapSplitMeta(
            split=str(payload.get('split', self.split_dir.name)),
            num_examples=int(payload['num_examples']),
            tokens_per_example=int(payload.get('tokens_per_example', payload.get('block_size', 1024))),
            bars_per_sample=int(payload.get('bars_per_sample', 8)),
            num_attributes=4,
            pad_id=int(payload['token_ids']['pad_id']),
            bar_id=int(payload['token_ids']['bar_id']),
        )

        self.x_path = self.split_dir / 'X.u16.memmap'
        self.bi_path = self.split_dir / 'bar_indices.u8.memmap'
        self.a_path = self.split_dir / 'attributes.u8.memmap'

        self._pid: Optional[int] = None
        self._x = None
        self._bi = None
        self._a = None

    def __len__(self) -> int:
        return self.meta.num_examples

    def _ensure_open(self) -> None:
        pid = os.getpid()
        if self._pid == pid and self._x is not None:
            return

        n = self.meta.num_examples
        t = self.meta.tokens_per_example
        b = self.meta.bars_per_sample
        a = self.meta.num_attributes

        self._x = np.memmap(self.x_path, mode='r', dtype=np.uint16, shape=(n, t))
        self._bi = np.memmap(self.bi_path, mode='r', dtype=np.uint8, shape=(n, t))
        self._a = np.memmap(self.a_path, mode='r', dtype=np.uint8, shape=(n, b, a))
        self._pid = pid

    def __getitem__(self, idx: int):
        self._ensure_open()
        return self._x[idx], self._bi[idx], self._a[idx]

    def __len__(self) -> int:
        return self.meta.num_examples

    def _ensure_open(self) -> None:
        pid = os.getpid()
        if self._pid == pid and self._x is not None:
            return

        n = self.meta.num_examples
        t = self.meta.tokens_per_example
        b = self.meta.bars_per_sample
        a = self.meta.num_attributes

        self._x = np.memmap(self.x_path, mode='r', dtype=np.uint16, shape=(n, t))
        self._bi = np.memmap(self.bi_path, mode='r', dtype=np.uint8, shape=(n, t))
        self._a = np.memmap(self.a_path, mode='r', dtype=np.uint8, shape=(n, b, a))
        self._pid = pid

    def __getitem__(self, idx: int):
        self._ensure_open()
        return self._x[idx], self._bi[idx], self._a[idx]


def make_collate_fn(*, pad_id: int) -> Callable:
    def collate(batch):
        xs, bis, attrs = zip(*batch)
        x_np = np.stack(xs, axis=0)
        bi_np = np.stack(bis, axis=0)
        a_np = np.stack(attrs, axis=0)

        X = torch.from_numpy(x_np.astype(np.int64, copy=False))
        bar_indices = torch.from_numpy(bi_np.astype(np.int64, copy=False))
        attributes = torch.from_numpy(a_np.astype(np.int64, copy=False))

        Y = torch.empty_like(X)
        Y[:, :-1] = X[:, 1:]
        Y[:, -1] = int(pad_id)

        return {'X': X, 'Y': Y, 'bar_indices': bar_indices, 'attributes': attributes}

    return collate
'''

# --- Grammar + sampling + generation (verbatim from repo) ---
GRAMMAR_CODE = r'''
from dataclasses import dataclass
from typing import Iterable, Literal

import torch


TokenType = Literal[
    'PAD',
    'BOS',
    'Bar',
    'Position',
    'Tempo',
    'Pitch',
    'Velocity',
    'Duration',
    'Other',
]


def token_type(token_str: str) -> TokenType:
    head = token_str.split('_', 1)[0]
    if head == 'PAD':
        return 'PAD'
    if head == 'BOS':
        return 'BOS'
    if head == 'Bar':
        return 'Bar'
    if head == 'Position':
        return 'Position'
    if head == 'Tempo':
        return 'Tempo'
    if head == 'Pitch':
        return 'Pitch'
    if head == 'Velocity':
        return 'Velocity'
    if head == 'Duration':
        return 'Duration'
    return 'Other'


@dataclass(frozen=True)
class GrammarVocabIndex:
    ids_by_type: dict[TokenType, list[int]]


def build_vocab_index(id_to_token: dict[int, str]) -> GrammarVocabIndex:
    ids_by_type: dict[TokenType, list[int]] = {
        'PAD': [],
        'BOS': [],
        'Bar': [],
        'Position': [],
        'Tempo': [],
        'Pitch': [],
        'Velocity': [],
        'Duration': [],
        'Other': [],
    }
    for tid, s in id_to_token.items():
        ids_by_type[token_type(s)].append(int(tid))
    for k in ids_by_type:
        ids_by_type[k] = sorted(ids_by_type[k])
    return GrammarVocabIndex(ids_by_type=ids_by_type)


def allowed_next_ids(
    prev_token_id: int,
    *,
    id_to_token: dict[int, str],
    vocab_index: GrammarVocabIndex,
    bar_id: int,
    pad_id: int,
    bos_id: int,
    allow_bar: bool = True,
) -> list[int]:
    prev_str = id_to_token[int(prev_token_id)]
    prev_type = token_type(prev_str)

    if prev_token_id == int(bos_id) or prev_type == 'BOS':
        return [int(bar_id)]

    if prev_token_id == int(bar_id) or prev_type == 'Bar':
        return vocab_index.ids_by_type['Position']

    if prev_type == 'Pitch':
        return vocab_index.ids_by_type['Velocity']

    if prev_type == 'Velocity':
        return vocab_index.ids_by_type['Duration']

    if prev_type == 'Tempo':
        return vocab_index.ids_by_type['Position']

    if prev_type == 'Position':
        allowed: list[int] = []
        allowed += vocab_index.ids_by_type['Pitch']
        allowed += vocab_index.ids_by_type['Tempo']
        if allow_bar:
            allowed.append(int(bar_id))
        return sorted(set(allowed))

    if prev_type == 'Duration':
        allowed = []
        allowed += vocab_index.ids_by_type['Pitch']
        allowed += vocab_index.ids_by_type['Position']
        allowed += vocab_index.ids_by_type['Tempo']
        if allow_bar:
            allowed.append(int(bar_id))
        return sorted(set(allowed))

    allowed = []
    allowed += vocab_index.ids_by_type['Bar'] if allow_bar else []
    allowed += vocab_index.ids_by_type['Position']
    allowed += vocab_index.ids_by_type['Tempo']
    allowed += vocab_index.ids_by_type['Pitch']
    allowed = [i for i in allowed if i not in (int(pad_id), int(bos_id))]
    return sorted(set(allowed))


def mask_logits_inplace(
    logits: torch.Tensor,
    *,
    allowed_ids: Iterable[int],
) -> torch.Tensor:
    if logits.ndim != 1:
        raise ValueError(f'Expected logits [vocab], got {tuple(logits.shape)}')
    allowed = torch.tensor(list(allowed_ids), device=logits.device, dtype=torch.long)
    mask = torch.ones((logits.size(0),), device=logits.device, dtype=torch.bool)
    mask[allowed] = False
    logits[mask] = float('-inf')
    return logits
'''

SAMPLE_CODE = r'''
from typing import Optional

import torch


def sample_next_token(
    logits: torch.Tensor,
    *,
    top_p: float = 0.9,
    temperature: float = 1.0,
    generator: Optional[torch.Generator] = None,
) -> int:
    if logits.ndim != 1:
        raise ValueError(f'Expected logits shape [vocab], got {tuple(logits.shape)}')
    if not (0.0 < float(top_p) <= 1.0):
        raise ValueError('top_p must be in (0, 1].')
    if float(temperature) <= 0.0:
        raise ValueError('temperature must be > 0.')

    if float(temperature) != 1.0:
        logits = logits / float(temperature)

    probs = torch.softmax(logits - logits.max(), dim=-1)

    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cdf = torch.cumsum(sorted_probs, dim=-1)
    keep = cdf <= float(top_p)
    keep[0] = True
    if keep.sum().item() < keep.numel():
        first_excluded = int(keep.sum().item())
        keep[first_excluded] = True

    filt_probs = sorted_probs[keep]
    filt_idx = sorted_idx[keep]
    filt_probs = filt_probs / filt_probs.sum()

    choice = torch.multinomial(filt_probs, num_samples=1, replacement=False, generator=generator)
    next_id = int(filt_idx[int(choice.item())].item())
    return next_id
'''

GENERATE_CODE = r'''
from dataclasses import dataclass
from typing import Literal, Optional

import torch

from musicgen_kaggle_eval_all_min import build_vocab_index, allowed_next_ids, mask_logits_inplace
from musicgen_kaggle_eval_all_min import sample_next_token


ModelType = Literal['plain', 'vae', 'simple_vae']


@dataclass(frozen=True)
class GenerationConfig:
    block_size: int = 1024
    bars_per_sample: int = 8
    temperature: float = 1.0
    top_p: float = 0.9


def compute_bar_indices_prefix(tokens: list[int], *, bar_id: int, bars_per_sample: int) -> list[int]:
    cur = -1
    max_bar = bars_per_sample - 1
    out: list[int] = []
    for tid in tokens:
        if int(tid) == int(bar_id):
            cur += 1
        if cur < 0:
            cur = 0
        if cur > max_bar:
            cur = max_bar
        out.append(int(cur))
    return out


@torch.no_grad()
def generate_remi_tokens(
    *,
    model_type: ModelType,
    model,
    attributes: torch.Tensor,
    pad_id: int,
    bos_id: int,
    bar_id: int,
    id_to_token: dict[int, str],
    cfg: GenerationConfig,
    device: torch.device,
    z_p: Optional[torch.Tensor] = None,
    seed: Optional[int] = None,
    vocab_index=None,
) -> list[int]:
    if attributes.ndim != 3 or attributes.shape[0] != 1 or attributes.shape[1] != cfg.bars_per_sample or attributes.shape[2] != 4:
        raise ValueError(f'attributes must have shape [1,{cfg.bars_per_sample},4], got {tuple(attributes.shape)}')

    model.eval()
    model.to(device)

    gen = torch.Generator(device=device)
    if seed is not None:
        gen.manual_seed(int(seed))

    if vocab_index is None:
        vocab_index = build_vocab_index(id_to_token)

    tokens: list[int] = [int(bos_id)]
    bar_count = 0

    X_full = torch.full((1, cfg.block_size), int(pad_id), dtype=torch.long, device=device)

    z_p_in = None
    if model_type in ('vae', 'simple_vae'):
        if z_p is None:
            z_p_in = torch.randn((1, getattr(model.cfg, 'z_dim', 128)), generator=gen, device=device)
        else:
            z_p_in = z_p.to(device=device, dtype=torch.float32)
            if z_p_in.ndim != 2 or z_p_in.shape[0] != 1:
                raise ValueError(f'z_p must have shape [1,z_dim], got {tuple(z_p_in.shape)}')

    for _t in range(cfg.block_size - 1):
        X_full.fill_(int(pad_id))
        X_full[0, : len(tokens)] = torch.tensor(tokens, dtype=torch.long, device=device)

        bar_idx_prefix = compute_bar_indices_prefix(tokens, bar_id=bar_id, bars_per_sample=cfg.bars_per_sample)
        cur_bar = int(bar_idx_prefix[-1]) if bar_idx_prefix else 0
        bar_indices = torch.full((1, cfg.block_size), cur_bar, dtype=torch.long, device=device)
        bar_indices[0, : len(bar_idx_prefix)] = torch.tensor(bar_idx_prefix, dtype=torch.long, device=device)

        if model_type == 'plain':
            out = model(X_full, bar_indices=bar_indices, attributes=attributes)
        elif model_type == 'simple_vae':
            out = model(X_full, bar_indices=bar_indices, attributes=attributes, z_p=z_p_in)
        elif model_type == 'vae':
            out = model(X_full, bar_indices=bar_indices, attributes=attributes, z_p=z_p_in)
        else:
            raise ValueError(f'Unknown model_type: {model_type}')

        logits = out['logits']
        step_logits = logits[0, len(tokens) - 1].float().clone()

        prev_id = int(tokens[-1])
        allowed = allowed_next_ids(
            prev_id,
            id_to_token=id_to_token,
            vocab_index=vocab_index,
            bar_id=bar_id,
            pad_id=pad_id,
            bos_id=bos_id,
            allow_bar=True,
        )
        mask_logits_inplace(step_logits, allowed_ids=allowed)

        next_id = sample_next_token(step_logits, top_p=cfg.top_p, temperature=cfg.temperature, generator=gen)

        if int(next_id) == int(bar_id) and bar_count >= cfg.bars_per_sample:
            break

        tokens.append(int(next_id))
        if int(next_id) == int(bar_id):
            bar_count += 1

    if len(tokens) < cfg.block_size:
        tokens = tokens + [int(pad_id)] * (cfg.block_size - len(tokens))
    else:
        tokens = tokens[: cfg.block_size]

    return tokens
'''

# --- Attribute recomputation + binning (from control_eval.py, trimmed) ---
ATTR_CODE = r'''
import json
from pathlib import Path

import numpy as np

ATTRIBUTE_NAMES = ['polyphony_rate', 'rhythmic_intensity', 'velocity_dynamics', 'note_density']

BLOCK_SIZE = 1024
BARS_PER_SAMPLE = 8


def _parse_duration_beats(token: str) -> float:
    raw = token.split('_', 1)[1]
    try:
        return float(raw)
    except Exception:
        parts = raw.split('.')
        if len(parts) == 3:
            a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
            if c == 0:
                raise ValueError(f'Cannot parse duration token: {token}')
            return float(a + (b / c))
        raise ValueError(f'Cannot parse duration token: {token}')


def compute_raw_attributes_per_bar_from_tokens(
    x_tokens_1024: list[int],
    *,
    id_to_token: dict[int, str],
    bar_token_id: int,
    bars_per_sample: int = BARS_PER_SAMPLE,
) -> list[list[float]]:
    if len(x_tokens_1024) != BLOCK_SIZE:
        raise ValueError(f'Expected {BLOCK_SIZE} tokens, got {len(x_tokens_1024)}')

    onsets = [0 for _ in range(bars_per_sample)]
    pos_has_onset: list[set[int]] = [set() for _ in range(bars_per_sample)]
    velocities: list[list[int]] = [[] for _ in range(bars_per_sample)]
    global_intervals: list[tuple[float, float]] = []

    cur_bar = -1
    cur_pos = 0

    def parse_pos(tok: str) -> int:
        return int(tok.split('_', 1)[1])

    def parse_vel(tok: str) -> int:
        return int(tok.split('_', 1)[1])

    i = 0
    while i < len(x_tokens_1024):
        tid = x_tokens_1024[i]
        tok = id_to_token.get(int(tid))
        if tok is None:
            i += 1
            continue
        typ = tok.split('_', 1)[0]

        if tid == bar_token_id or typ == 'Bar':
            cur_bar += 1
            cur_pos = 0
            i += 1
            continue

        if cur_bar < 0:
            i += 1
            continue
        if cur_bar >= bars_per_sample:
            break

        if typ == 'Position':
            cur_pos = parse_pos(tok)
            i += 1
            continue

        if typ == 'Pitch' and i + 2 < len(x_tokens_1024):
            vtok = id_to_token.get(int(x_tokens_1024[i + 1]), '')
            dtok = id_to_token.get(int(x_tokens_1024[i + 2]), '')
            if vtok.startswith('Velocity_') and dtok.startswith('Duration_'):
                vel = parse_vel(vtok)
                dur = _parse_duration_beats(dtok)

                onsets[cur_bar] += 1
                pos_has_onset[cur_bar].add(cur_pos)
                velocities[cur_bar].append(vel)

                s = cur_bar * 4.0 + (cur_pos * 0.25)
                e = s + dur
                global_intervals.append((s, e))

                i += 3
                continue

        i += 1

    attrs: list[list[float]] = []
    for b in range(bars_per_sample):
        note_density = float(onsets[b])
        rhythmic_intensity = float(len(pos_has_onset[b]) / 16.0)
        vel_std = float(np.std(np.asarray(velocities[b], dtype=float), ddof=0)) if velocities[b] else 0.0

        poly_samples: list[int] = []
        for p in range(16):
            t = b * 4.0 + (p * 0.25)
            active = 0
            for s, e in global_intervals:
                if s <= t < e:
                    active += 1
            poly_samples.append(active)
        polyphony_rate = float(sum(poly_samples) / 16.0)

        attrs.append([polyphony_rate, rhythmic_intensity, vel_std, note_density])
    return attrs


def bin_attributes(
    raw_attrs_barsx4: list[list[float]],
    thresholds: dict[str, list[float]],
    bars_per_sample: int = BARS_PER_SAMPLE,
) -> list[list[int]]:
    arr = np.asarray(raw_attrs_barsx4, dtype=float)
    if arr.ndim != 2 or arr.shape != (bars_per_sample, 4):
        raise ValueError(f'Expected shape [{bars_per_sample},4], got {arr.shape}')
    out = np.zeros_like(arr, dtype=int)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        th = np.asarray(thresholds[name], dtype=float)
        if th.shape != (7,):
            raise ValueError(f'Expected 7 thresholds for {name}, got shape {th.shape}')
        out[:, j] = np.digitize(arr[:, j], th, right=False)
    return out.astype(int).tolist()


def load_thresholds(path: str) -> dict[str, list[float]]:
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    th = payload['thresholds']
    for name in ATTRIBUTE_NAMES:
        if name not in th:
            raise ValueError(f'Threshold for {name!r} missing in {path}')
    return th
'''

MODULE_CODE = (
    PLAIN_MODEL_CODE
    + "\n\n"
    + SIMPLE_MODEL_CODE
    + "\n\n"
    + VAE_MODEL_CODE
    + "\n\n"
    + DATA_CODE
    + "\n\n"
    + GRAMMAR_CODE
    + "\n\n"
    + SAMPLE_CODE
    + "\n\n"
    + ATTR_CODE
    + "\n\n"
    + GENERATE_CODE
)

MODULE_PATH.write_text(MODULE_CODE, encoding='utf-8')
print('Wrote module:', MODULE_PATH)

if str(Path('.').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('.').resolve()))

import musicgen_kaggle_eval_all_min as mg
print('Imported self-contained module:', mg.__file__)


In [ ]:
# -------------------------
# Load tokenizer metadata + thresholds
# -------------------------

tok_meta = json.loads(TOKENIZER_META_PATH.read_text(encoding='utf-8'))

# expected keys: id_to_token, pad_id, bos_id, bar_id
id_to_token = {int(k): v for k, v in tok_meta['id_to_token'].items()} if isinstance(tok_meta['id_to_token'], dict) else {int(i): s for i, s in enumerate(tok_meta['id_to_token'])}

pad_id = int(tok_meta.get('pad_id', -1))
bos_id = int(tok_meta.get('bos_id', -1))
bar_id = int(tok_meta.get('bar_id', -1))

if pad_id < 0 or bos_id < 0 or bar_id < 0:
    # fallback to meta.json contract
    meta_json = json.loads((Path(TEST_DIR) / 'meta.json').read_text(encoding='utf-8'))
    pad_id = int(meta_json['pad_id'])
    bos_id = int(meta_json['bos_id'])
    bar_id = int(meta_json['bar_id'])

thresholds = json.loads(QUANTILES_PATH.read_text(encoding='utf-8'))['thresholds']

vocab_index = mg.build_vocab_index(id_to_token)

print('pad_id:', pad_id, 'bos_id:', bos_id, 'bar_id:', bar_id)
print('vocab size:', len(id_to_token))
print('threshold keys:', list(thresholds.keys()))
print('built vocab_index')


In [ ]:
# -------------------------
# DataLoader
# -------------------------

from torch.utils.data import DataLoader

test_ds = mg.MemmapMusicDataset(TEST_DIR)
collate_fn = mg.make_collate_fn(pad_id=pad_id)

test_dl_e1 = DataLoader(
    test_ds,
    batch_size=E1_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
    collate_fn=collate_fn,
)

print('test meta:', test_ds.meta)


In [ ]:
# -------------------------
# Load checkpoints + instantiate models
# -------------------------

def _load_ckpt(path: Path) -> dict:
    ckpt = torch.load(path, map_location='cpu')
    if isinstance(ckpt, dict) and 'cfg' in ckpt:
        return ckpt
    raise ValueError(f'Unexpected ckpt format at {path}')


def _instantiate_plain(ckpt: dict):
    cfg = mg.PlainTransformerConfig(**ckpt['cfg']) if isinstance(ckpt['cfg'], dict) else ckpt['cfg']
    m = mg.PlainTransformerDecoder(cfg)
    sd = ckpt.get('model') or ckpt.get('model_state_dict') or ckpt.get('state_dict')
    if sd is None:
        raise KeyError('ckpt missing model state dict')
    m.load_state_dict(sd, strict=True)
    return m


def _instantiate_simple(ckpt: dict):
    cfg = mg.SimpleVAEConfig(**ckpt['cfg']) if isinstance(ckpt['cfg'], dict) else ckpt['cfg']
    m = mg.SimpleMusicVAE(cfg)
    sd = ckpt.get('model') or ckpt.get('model_state_dict') or ckpt.get('state_dict')
    if sd is None:
        raise KeyError('ckpt missing model state dict')
    m.load_state_dict(sd, strict=True)
    return m


def _instantiate_vae(ckpt: dict):
    cfg = mg.VAEConfig(**ckpt['cfg']) if isinstance(ckpt['cfg'], dict) else ckpt['cfg']
    m = mg.MusicVAE(cfg)
    sd = ckpt.get('model') or ckpt.get('model_state_dict') or ckpt.get('state_dict')
    if sd is None:
        raise KeyError('ckpt missing model state dict')
    m.load_state_dict(sd, strict=True)
    return m

plain_ckpt = _load_ckpt(PLAIN_CKPT)
simple_ckpt = _load_ckpt(SIMPLE_CKPT)
vae_ckpt = _load_ckpt(VAE_CKPT)

plain_model = _instantiate_plain(plain_ckpt).to(DEVICE)
simple_model = _instantiate_simple(simple_ckpt).to(DEVICE)
vae_model = _instantiate_vae(vae_ckpt).to(DEVICE)

for m in [plain_model, simple_model, vae_model]:
    m.eval()

print('loaded models')
print('plain cfg:', getattr(plain_model, 'cfg', None))
print('simple cfg:', getattr(simple_model, 'cfg', None))
print('vae cfg:', getattr(vae_model, 'cfg', None))


In [ ]:
# -------------------------
# Sanity check: 1 forward pass per model
# -------------------------

b = next(iter(test_dl_e1))
X = b['X'].to(DEVICE)
Y = b['Y'].to(DEVICE)
bar_indices = b['bar_indices'].to(DEVICE)
attributes = b['attributes'].to(DEVICE)

with torch.no_grad():
    out_plain = plain_model(X, bar_indices=bar_indices, attributes=attributes, targets=Y, pad_id=pad_id)
    out_simple = simple_model(X, bar_indices=bar_indices, attributes=attributes, targets=Y, pad_id=pad_id)
    out_vae = vae_model(X, bar_indices=bar_indices, attributes=attributes, targets=Y, pad_id=pad_id)

print('plain logits:', tuple(out_plain['logits'].shape), 'loss:', float(out_plain['loss']))
print('simple logits:', tuple(out_simple['logits'].shape), 'loss_recon:', float(out_simple['loss_recon']), 'loss_kl:', float(out_simple['loss_kl']))
print('vae logits:', tuple(out_vae['logits'].shape), 'loss_recon:', float(out_vae['loss_recon']), 'loss_kl:', float(out_vae['loss_kl']))


In [ ]:
# -------------------------
# Helpers for metrics + console progress
# -------------------------

# If you truly want a line printed for every loop iteration, keep at 1.
# (This can be extremely verbose in Kaggle outputs.)
PRINT_EVERY = 1


def _fmt_secs(s: float) -> str:
    s = max(0.0, float(s))
    if s < 60:
        return f"{s:5.1f}s"
    m = s / 60.0
    if m < 60:
        return f"{m:5.1f}m"
    h = m / 60.0
    return f"{h:5.2f}h"


def print_progress(*, tag: str, i: int, total: int, start_time: float, every: int = 1) -> None:
    if every <= 0:
        return
    if i % every != 0 and i != total:
        return
    now = time.time()
    elapsed = now - start_time
    rate = elapsed / max(1, i)
    eta = rate * max(0, total - i)
    print(f"[{tag}] {i}/{total}  elapsed={_fmt_secs(elapsed)}  eta={_fmt_secs(eta)}")


def ce_and_ppl_from_logits(logits: torch.Tensor, targets: torch.Tensor, *, pad_id: int) -> tuple[float, float, int]:
    # logits: [B,T,V], targets: [B,T]
    V = logits.size(-1)
    loss = F.cross_entropy(
        logits.view(-1, V),
        targets.view(-1),
        ignore_index=int(pad_id),
        reduction='sum',
    )
    n = int((targets != int(pad_id)).sum().item())
    ce = float((loss / max(1, n)).item())
    ppl = float(math.exp(min(20.0, ce)))
    return ce, ppl, n


def kl_bits_per_sample(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    # standard KL(q||p) for diagonal Gaussian, in bits, shape [B]
    kl_nats = 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar).sum(dim=-1)
    return kl_nats / math.log(2.0)


def pearsonr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x - x.mean()
    y = y - y.mean()
    denom = (np.sqrt((x * x).sum()) * np.sqrt((y * y).sum()))
    if denom <= 0:
        return float('nan')
    return float((x * y).sum() / denom)


def _rankdata(a: np.ndarray) -> np.ndarray:
    # average ranks for ties
    a = np.asarray(a)
    order = np.argsort(a)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(a), dtype=float)
    # tie handling
    uniq, idx_start, counts = np.unique(a[order], return_index=True, return_counts=True)
    for s, c in zip(idx_start, counts):
        if c > 1:
            inds = order[s : s + c]
            ranks[inds] = ranks[inds].mean()
    return ranks


def spearmanr(x: np.ndarray, y: np.ndarray) -> float:
    return pearsonr(_rankdata(np.asarray(x)), _rankdata(np.asarray(y)))


def confusion_from_bins(requested: list[int], measured: list[int], n_bins: int = 8) -> list[list[int]]:
    m = [[0 for _ in range(n_bins)] for __ in range(n_bins)]
    for r, t in zip(requested, measured):
        m[int(r)][int(t)] += 1
    return m


In [ ]:
# -------------------------
# E1 — Reconstruction + KL usage
# -------------------------

def run_e1(model_name: str, model, *, is_vae: bool) -> dict:
    ce_sum = 0.0
    ppl_sum = 0.0
    n_tok_sum = 0

    kl_bits_sum = 0.0
    n_ex_sum = 0
    kl_dim_sum = None

    n_batches = 0
    start = time.time()
    total = int(min(E1_MAX_BATCHES, len(test_dl_e1)))
    pbar = tqdm(test_dl_e1, total=total, desc=f'E1 {model_name}')
    for batch in pbar:
        if n_batches >= E1_MAX_BATCHES:
            break
        n_batches += 1
        print_progress(tag=f'E1 {model_name}', i=n_batches, total=total, start_time=start, every=PRINT_EVERY)

        X = batch['X'].to(DEVICE)
        Y = batch['Y'].to(DEVICE)
        bi = batch['bar_indices'].to(DEVICE)
        a = batch['attributes'].to(DEVICE)

        with torch.no_grad():
            out = model(X, bar_indices=bi, attributes=a, targets=Y, pad_id=pad_id)
            logits = out['logits']

            ce, ppl, n = ce_and_ppl_from_logits(logits, Y, pad_id=pad_id)
            ce_sum += ce * n
            ppl_sum += ppl * n
            n_tok_sum += n

            if is_vae:
                mu = out['mu']
                logvar = out['logvar']
                kl_bits = kl_bits_per_sample(mu, logvar)
                kl_bits_sum += float(kl_bits.sum().item())
                n_ex_sum += int(kl_bits.numel())

                kl_dim = (0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar)).mean(dim=0) / math.log(2.0)
                if kl_dim_sum is None:
                    kl_dim_sum = kl_dim.detach().cpu().numpy()
                else:
                    kl_dim_sum = kl_dim_sum + kl_dim.detach().cpu().numpy()

        pbar.set_postfix({'ce': ce_sum / max(1, n_tok_sum), 'tok': n_tok_sum})

    out = {
        'model': model_name,
        'ce': float(ce_sum / max(1, n_tok_sum)),
        'ppl': float(ppl_sum / max(1, n_tok_sum)),
        'n_tokens': int(n_tok_sum),
        'batches': int(n_batches),
    }

    if is_vae:
        out['kl_bits_per_sample'] = float(kl_bits_sum / max(1, n_ex_sum))
        kl_dim_mean_bits = (kl_dim_sum / max(1, n_batches)).tolist() if kl_dim_sum is not None else None
        out['kl_dim_mean_bits'] = kl_dim_mean_bits
        if kl_dim_sum is not None:
            out['active_units_gt_1bit'] = int((np.asarray(kl_dim_mean_bits) > 1.0).sum())

    return out


e1_plain = run_e1('plain', plain_model, is_vae=False)
e1_simple = run_e1('simple', simple_model, is_vae=True)
e1_vae = run_e1('vae', vae_model, is_vae=True)

e1 = {'plain': e1_plain, 'simple': e1_simple, 'vae': e1_vae}
(OUT_DIR / 'e1_recon_kl.json').write_text(json.dumps(e1, indent=2), encoding='utf-8')
print('Wrote', OUT_DIR / 'e1_recon_kl.json')


In [ ]:
# -------------------------
# E2 — Attribute controllability (generation + attribute recomputation)
# -------------------------

ATTRIBUTE_NAMES = ['polyphony_rate', 'rhythmic_intensity', 'velocity_dynamics', 'note_density']


def mean_raw4(raw_barsx4: list[list[float]]) -> np.ndarray:
    arr = np.asarray(raw_barsx4, dtype=float)
    return arr.mean(axis=0)


def bin_raw4(raw4: np.ndarray, thresholds: dict[str, list[float]]) -> list[int]:
    out = []
    for j, name in enumerate(ATTRIBUTE_NAMES):
        th = np.asarray(thresholds[name], dtype=float)
        out.append(int(np.digitize(float(raw4[j]), th, right=False)))
    return out


def run_e2_for_model(model_name: str, model, *, model_type: str) -> dict:
    gen_cfg = mg.GenerationConfig(
        block_size=int(test_ds.meta.tokens_per_example),
        bars_per_sample=int(test_ds.meta.bars_per_sample),
        temperature=float(E2_TEMPERATURE),
        top_p=float(E2_TOP_P),
    )

    rng = np.random.default_rng(int(E2_SEED))

    results = {}

    fixed_z = None
    if model_type in ('vae', 'simple_vae'):
        zdim = int(getattr(model.cfg, 'z_dim', 128))
        fixed_z = torch.from_numpy(rng.standard_normal((1, zdim)).astype(np.float32))

    # --- quick timing probe (1 generation) ---
    probe_attrs = np.full((1, int(test_ds.meta.bars_per_sample), 4), int(E2_BASELINE_BIN), dtype=np.int64)
    probe_attrs_t = torch.from_numpy(probe_attrs).to(DEVICE)
    t0 = time.time()
    _ = mg.generate_remi_tokens(
        model_type=model_type,
        model=model,
        attributes=probe_attrs_t,
        pad_id=int(pad_id),
        bos_id=int(bos_id),
        bar_id=int(bar_id),
        id_to_token=id_to_token,
        cfg=gen_cfg,
        device=DEVICE,
        z_p=fixed_z,
        seed=int(E2_SEED),
        vocab_index=vocab_index,
    )
    sec_per_sample = max(1e-3, time.time() - t0)

    # total samples per model = 4 attrs * 8 bins * N_PER_BIN
    max_samples = int(E2_TIME_BUDGET_SECS / sec_per_sample)
    target_n = int(max_samples // (4 * 8))
    n_per_bin = int(max(E2_MIN_N_PER_BIN, min(E2_N_PER_BIN, target_n)))

    print(f"[E2 {model_name}] probe sec/sample={sec_per_sample:.3f} -> n_per_bin={n_per_bin} (budget={E2_TIME_BUDGET_SECS}s)")

    for aj, aname in enumerate(ATTRIBUTE_NAMES):
        requested_bins = []
        measured_bins = []
        measured_raw = []

        # total inner generations for ETA
        total_gens = int(8 * n_per_bin)
        start_attr = time.time()
        done = 0

        for target_bin in range(8):
            # uniform attributes across 8 bars
            attrs = np.full((1, int(test_ds.meta.bars_per_sample), 4), int(E2_BASELINE_BIN), dtype=np.int64)
            attrs[:, :, aj] = int(target_bin)
            attrs_t = torch.from_numpy(attrs).to(DEVICE)

            for k in tqdm(range(n_per_bin), desc=f'E2 {model_name} {aname} bin={target_bin}', leave=False):
                done += 1
                print_progress(tag=f'E2 {model_name} {aname}', i=done, total=total_gens, start_time=start_attr, every=PRINT_EVERY)
                tokens = mg.generate_remi_tokens(
                    model_type=model_type,
                    model=model,
                    attributes=attrs_t,
                    pad_id=int(pad_id),
                    bos_id=int(bos_id),
                    bar_id=int(bar_id),
                    id_to_token=id_to_token,
                    cfg=gen_cfg,
                    device=DEVICE,
                    z_p=fixed_z,
                    seed=int(E2_SEED + 1000 * aj + 10 * target_bin + k),
                    vocab_index=vocab_index,
                )

                raw = mg.compute_raw_attributes_per_bar_from_tokens(
                    tokens,
                    id_to_token=id_to_token,
                    bar_token_id=int(bar_id),
                    bars_per_sample=int(test_ds.meta.bars_per_sample),
                )
                raw4 = mean_raw4(raw)
                b4 = bin_raw4(raw4, thresholds)

                requested_bins.append(int(target_bin))
                measured_bins.append(int(b4[aj]))
                measured_raw.append(float(raw4[aj]))

        conf = confusion_from_bins(requested_bins, measured_bins, n_bins=8)
        req = np.asarray(requested_bins)
        meas = np.asarray(measured_bins)
        exact = float((req == meas).mean())
        pm1 = float((np.abs(req - meas) <= 1).mean())
        pear = pearsonr(req, np.asarray(measured_raw))
        spear = spearmanr(req, np.asarray(measured_raw))

        results[aname] = {
            'n_total': int(len(requested_bins)),
            'accuracy_exact': exact,
            'accuracy_pm1': pm1,
            'pearson_req_vs_raw': pear,
            'spearman_req_vs_raw': spear,
            'confusion': conf,
        }

    return {'model': model_name, 'per_attribute': results}


e2 = {
    'plain': run_e2_for_model('plain', plain_model, model_type='plain'),
    'simple': run_e2_for_model('simple', simple_model, model_type='simple_vae'),
    'vae': run_e2_for_model('vae', vae_model, model_type='vae'),
}

(OUT_DIR / 'e2_controllability.json').write_text(json.dumps(e2, indent=2), encoding='utf-8')
print('Wrote', OUT_DIR / 'e2_controllability.json')


In [ ]:
# -------------------------
# E3 — Latent μ extraction + PCA/UMAP + kNN purity
# -------------------------

from torch.utils.data import DataLoader

vae_dl = DataLoader(
    test_ds,
    batch_size=E3_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=False,
    collate_fn=collate_fn,
)

mus = []
labels = []

n_seen = 0
start = time.time()
total_batches = int(math.ceil(E3_N_LATENT / E3_BATCH_SIZE))
pbar = tqdm(vae_dl, desc='E3 encode', total=total_batches)
for bi, batch in enumerate(pbar, start=1):
    print_progress(tag='E3 encode', i=bi, total=total_batches, start_time=start, every=PRINT_EVERY)
    X = batch['X'].to(DEVICE)
    a = batch['attributes'].to(DEVICE)

    with torch.no_grad():
        mu, logvar = vae_model.encoder(X)

    mu_np = mu.detach().cpu().numpy()

    # label per sample: mean bin over bars for each attribute
    lab = a.detach().cpu().numpy().mean(axis=1)

    mus.append(mu_np)
    labels.append(lab)
    n_seen += mu_np.shape[0]
    if n_seen >= E3_N_LATENT:
        break

mu_all = np.concatenate(mus, axis=0)[:E3_N_LATENT]
lab_all = np.concatenate(labels, axis=0)[:E3_N_LATENT]

print('mu_all:', mu_all.shape, 'labels:', lab_all.shape)

# PCA 2D
mu_center = mu_all - mu_all.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(mu_center, full_matrices=False)
pca2 = (U[:, :2] * S[:2])

# UMAP optional
umap2 = None
try:
    import umap

    um = umap.UMAP(n_components=2, n_neighbors=25, min_dist=0.1, metric='euclidean', random_state=0)
    umap2 = um.fit_transform(mu_all)
except Exception as e:
    print('UMAP not available, skipping. Error:', repr(e))

# kNN purity (simple): for each point, how many of k neighbors share same bin label (per attribute)

def knn_purity(emb: np.ndarray, lab: np.ndarray, *, k: int) -> dict:
    try:
        from sklearn.neighbors import NearestNeighbors

        nn = NearestNeighbors(n_neighbors=k + 1, algorithm='auto').fit(emb)
        d, idx = nn.kneighbors(emb)
        idx = idx[:, 1:]
    except Exception:
        # fallback brute force for smaller N
        N = emb.shape[0]
        idx = []
        for i in range(N):
            d = ((emb - emb[i]) ** 2).sum(axis=1)
            nn_i = np.argsort(d)[1 : k + 1]
            idx.append(nn_i)
        idx = np.stack(idx, axis=0)

    out = {}
    for j, name in enumerate(ATTRIBUTE_NAMES):
        y = lab[:, j].round().astype(int)
        neigh = y[idx]
        out[name] = float((neigh == y[:, None]).mean())
    return out

pur_pca = knn_purity(pca2, lab_all, k=E3_KNN_K)
pur_umap = knn_purity(umap2, lab_all, k=E3_KNN_K) if umap2 is not None else None

np.savez_compressed(OUT_DIR / 'e3_latent_mu.npz', mu=mu_all, labels=lab_all, pca2=pca2)
if umap2 is not None:
    np.savez_compressed(OUT_DIR / 'e3_umap.npz', umap2=umap2)

(OUT_DIR / 'e3_knn_purity.json').write_text(
    json.dumps({'pca2': pur_pca, 'umap2': pur_umap}, indent=2),
    encoding='utf-8',
)

print('Wrote E3 outputs to', OUT_DIR)


In [ ]:
# -------------------------
# E4 — Latent sampling / interpolation (VAE)
# -------------------------


def slerp(t: float, a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    a = a / (np.linalg.norm(a) + eps)
    b = b / (np.linalg.norm(b) + eps)
    dot = np.clip(np.dot(a, b), -1.0, 1.0)
    omega = math.acos(dot)
    if abs(omega) < 1e-6:
        return (1.0 - t) * a + t * b
    so = math.sin(omega)
    return (math.sin((1.0 - t) * omega) / so) * a + (math.sin(t * omega) / so) * b


def generate_from_z(z: np.ndarray, attrs_bin: int = 3, *, seed: int) -> tuple[np.ndarray, list[int]]:
    attrs = np.full((1, int(test_ds.meta.bars_per_sample), 4), int(attrs_bin), dtype=np.int64)
    attrs_t = torch.from_numpy(attrs).to(DEVICE)

    z_t = torch.from_numpy(z.astype(np.float32)[None, :])

    gen_cfg = mg.GenerationConfig(
        block_size=int(test_ds.meta.tokens_per_example),
        bars_per_sample=int(test_ds.meta.bars_per_sample),
        temperature=float(E2_TEMPERATURE),
        top_p=float(E2_TOP_P),
    )

    tokens = mg.generate_remi_tokens(
        model_type='vae',
        model=vae_model,
        attributes=attrs_t,
        pad_id=int(pad_id),
        bos_id=int(bos_id),
        bar_id=int(bar_id),
        id_to_token=id_to_token,
        cfg=gen_cfg,
        device=DEVICE,
        z_p=z_t,
        seed=int(seed),
        vocab_index=vocab_index,
    )

    raw = mg.compute_raw_attributes_per_bar_from_tokens(
        tokens,
        id_to_token=id_to_token,
        bar_token_id=int(bar_id),
        bars_per_sample=int(test_ds.meta.bars_per_sample),
    )
    raw4 = mean_raw4(raw)
    return raw4, tokens


rng = np.random.default_rng(0)
zdim = int(getattr(vae_model.cfg, 'z_dim', 128))

anchors = rng.standard_normal((E4_N_ANCHORS, zdim)).astype(np.float32)

near_stats = []
interp_stats = []

start = time.time()
# total work units: for each anchor we do 1 base + NEAR + INTERP gens
per_anchor = 1 + int(E4_N_NEAR) + int(E4_N_INTERP)
total_units = int(E4_N_ANCHORS) * per_anchor
unit = 0

for i in tqdm(range(E4_N_ANCHORS), desc='E4 anchors'):
    z0 = anchors[i]

    unit += 1
    print_progress(tag='E4', i=unit, total=total_units, start_time=start, every=PRINT_EVERY)

    base_raw, _tok = generate_from_z(z0, attrs_bin=E2_BASELINE_BIN, seed=10000 + i)

    # neighborhood
    for j in range(E4_N_NEAR):
        unit += 1
        print_progress(tag='E4', i=unit, total=total_units, start_time=start, every=PRINT_EVERY)
        z = z0 + float(E4_SIGMA) * rng.standard_normal((zdim,)).astype(np.float32)
        raw, _ = generate_from_z(z, attrs_bin=E2_BASELINE_BIN, seed=20000 + 1000 * i + j)
        near_stats.append({'anchor': i, 'l2_z': float(np.linalg.norm(z - z0)), 'delta_raw_l2': float(np.linalg.norm(raw - base_raw))})

    # interpolation between z0 and a random other anchor
    k = int((i + 1) % E4_N_ANCHORS)
    z1 = anchors[k]
    for t_idx, t in enumerate(np.linspace(0.0, 1.0, E4_N_INTERP)):
        unit += 1
        print_progress(tag='E4', i=unit, total=total_units, start_time=start, every=PRINT_EVERY)
        zt = slerp(float(t), z0, z1).astype(np.float32)
        raw, _ = generate_from_z(zt, attrs_bin=E2_BASELINE_BIN, seed=30000 + 1000 * i + t_idx)
        interp_stats.append({'pair': [i, k], 't': float(t), 'raw4': raw.tolist()})


e4 = {
    'near_stats': near_stats,
    'interp_stats': interp_stats,
}

(OUT_DIR / 'e4_latent_sampling.json').write_text(json.dumps(e4, indent=2), encoding='utf-8')
print('Wrote', OUT_DIR / 'e4_latent_sampling.json')


In [ ]:
# -------------------------
# Summary artifact
# -------------------------

summary = {
    'e1': 'e1_recon_kl.json',
    'e2': 'e2_controllability.json',
    'e3_mu': 'e3_latent_mu.npz',
    'e3_umap': 'e3_umap.npz' if (OUT_DIR / 'e3_umap.npz').exists() else None,
    'e3_knn': 'e3_knn_purity.json',
    'e4': 'e4_latent_sampling.json',
}

(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Wrote', OUT_DIR / 'summary.json')
print('Done.')
